### **Proof-of-concept graph**
Uses 1 TF, ? genes\
Run file with 'python scripts/name.py'


Pipeline: ChIP-seq narrowPeak files -> TF-gene regulatory networkx graph.

Expects:
  - one narrowPeak file per TF, named like "<TFNAME>.narrowPeak"
  - a TSS bed file: chrom, tss_start, tss_end, gene_name, score, strand

Produces:
  - a networkx DiGraph: TF nodes -> gene nodes, edges weighted by
    signal strength and annotated with distance-to-TSS.

In [1]:
import glob
import os
import subprocess
import networkx as nx

### **STEP 1: Parse a single narrowPeak file into a list of peak dicts**
Read a narrowPeak file and return a filtered list of peaks.

WHY: narrowPeak files (especially raw MACS2 output, not IDR-processed)
contain many low-confidence peaks. We filter on qValue here so garbage
peaks don't become garbage edges later. qValue in the file is
-log10(q), so q < 0.05 corresponds to a threshold of 1.3.

In [2]:
def parse_narrowpeak(path, qvalue_threshold=1.3):
    peaks = []
    with open(path) as f:
        for line in f:
            cols = line.rstrip("\n").split("\t")
            chrom, start, end = cols[0], int(cols[1]), int(cols[2])
            signal_value = float(cols[6])
            qvalue = float(cols[8])

            if qvalue < qvalue_threshold:
                continue  # discard: not statistically confident enough

            # summit = absolute genomic position of strongest signal in peak
            # (more precise than using the peak midpoint)
            summit_offset = int(cols[9]) if cols[9] != "-1" else (end - start) // 2
            summit = start + summit_offset

            peaks.append({
                "chrom": chrom,
                "start": start,
                "end": end,
                "summit": summit,
                "signal_value": signal_value,
                "qvalue": qvalue,
            })
    return peaks


### **ML Model**
Added in later version

### **STEP 2: Load gene TSS reference table**
Load a TSS bed file into a list of gene dicts, one row per gene.

WHY: narrowPeak files carry zero gene information. This is the
independent reference table needed to translate "genomic coordinate"
into "gene identity". In practice this comes from a GENCODE GTF,
reduced down to one TSS coordinate per gene (start coord for +
strand genes, end coord for - strand genes).

In [4]:
def load_tss(tss_bed_path):
    genes = []
    with open(tss_bed_path) as f:
        for line in f:
            cols = line.rstrip("\n").split("\t")
            genes.append({
                "chrom": cols[0],
                "tss": int(cols[1]),
                "name": cols[3],
                "strand": cols[5],
            })
    return genes


### **STEP 3: Assign each peak to its nearest gene TSS**
For each peak, find the closest gene TSS on the same chromosome.

WHY: This is the actual "linear-distance regulatory assignment" step
discussed earlier -- the thing Hi-C would later refine. Doing it here
in pure Python (rather than shelling out to bedtools) keeps the
pipeline self-contained and easy to swap out later (e.g. replacing
this function's internals with a Hi-C-loop-based assignment is a
drop-in replacement -- same input/output shape).

NOTE: for large chromosome-scale datasets you'd want this to be sorted
+ a smarter nearest-neighbor search (or just call bedtools closest,
as shown earlier) rather than an O(n*m) scan. For 30-60 TFs at ChIP-seq
peak-count scale (thousands of peaks, ~20k genes) this brute-force
version is still fast enough to just work.

In [5]:
def assign_nearest_gene(peaks, genes):
    # group genes by chromosome so we don't compare against genes on
    # other chromosomes
    genes_by_chrom = {}
    for g in genes:
        genes_by_chrom.setdefault(g["chrom"], []).append(g)

    assignments = []
    for p in peaks:
        candidates = genes_by_chrom.get(p["chrom"], [])
        if not candidates:
            continue  # no genes annotated on this chromosome, skip peak

        # find gene with minimum distance from peak summit to TSS
        nearest = min(candidates, key=lambda g: abs(g["tss"] - p["summit"]))
        distance = abs(nearest["tss"] - p["summit"])

        assignments.append({
            "gene": nearest["name"],
            "signal_value": p["signal_value"],
            "qvalue": p["qvalue"],
            "distance_bp": distance,
        })
    return assignments


### **STEP 4: Build the graph across all TFs**
Loop over every narrowPeak file in narrowpeak_dir (one per TF),
assign peaks to genes, and accumulate everything into one
networkx DiGraph.

WHY a DiGraph: TF -> gene is directional (regulator -> target).
We also tag node types ("TF" or "gene") as node attributes so the
graph can be treated as bipartite/multipartite later even though
networkx doesn't enforce bipartite-ness structurally.

In [6]:
def build_graph(narrowpeak_dir, tss_bed_path, qvalue_threshold=1.3):
    genes = load_tss(tss_bed_path)
    G = nx.DiGraph()

    narrowpeak_files = glob.glob(os.path.join(narrowpeak_dir, "*.narrowPeak"))

    for path in narrowpeak_files:
        tf_name = os.path.splitext(os.path.basename(path))[0]

        peaks = parse_narrowpeak(path, qvalue_threshold=qvalue_threshold)
        assignments = assign_nearest_gene(peaks, genes)

        G.add_node(tf_name, node_type="TF")

        for a in assignments:
            gene = a["gene"]
            G.add_node(gene, node_type="gene")

            # WHY: one TF often has multiple peaks near the same gene
            # (e.g. multiple binding sites in a promoter/enhancer cluster).
            # We collapse these into a single edge, keeping the
            # strongest (max signal_value) peak's stats as the edge weight,
            # rather than creating parallel edges. This keeps downstream
            # graph metrics (degree, centrality) clean and interpretable.
            if G.has_edge(tf_name, gene):
                if a["signal_value"] > G[tf_name][gene]["weight"]:
                    G[tf_name][gene]["weight"] = a["signal_value"]
                    G[tf_name][gene]["distance_bp"] = a["distance_bp"]
                    G[tf_name][gene]["qvalue"] = a["qvalue"]
            else:
                G.add_edge(
                    tf_name, gene,
                    weight=a["signal_value"],
                    distance_bp=a["distance_bp"],
                    qvalue=a["qvalue"],
                )

    return G


### **<span style="color: red">-----------------------------------Main----------------------------------------</span>**
### **STEP 5: Run + check output**

In [ ]:
#Set working directory
#repo_root ="chipseq/"
#os.chdir(repo_root)

narrowpeak_dir = "../results/macs3/"  # name of folder with narrowpeak files. only those will be used so other stuff can be there too
tss_bed_path="../reference_data/2026-07-31_sorted-gencode-tss-protein-encoding.bed"
# First run: ../reference_data/2026-07-30_sorted-gencode-tss-with-ncRNA.bed
# Second run: ../reference_data/2026-07-31_sorted-gencode-tss-protein-encoding.bed

G = build_graph(
    narrowpeak_dir=narrowpeak_dir,
    tss_bed_path=tss_bed_path, 
    qvalue_threshold=1.3
)

print(f"Nodes: {G.number_of_nodes()}  Edges: {G.number_of_edges()}")

tf_nodes = [n for n, d in G.nodes(data=True) if d["node_type"] == "TF"]
gene_nodes = [n for n, d in G.nodes(data=True) if d["node_type"] == "gene"]
print(f"  TF nodes: {len(tf_nodes)}   Gene nodes: {len(gene_nodes)}")

print("\nEdges:")
for u, v, data in G.edges(data=True):
    print(f"  {u} -> {v}   weight={data['weight']:.2f}  "
            f"distance={data['distance_bp']}bp  qvalue={data['qvalue']:.2f}")


Nodes: 8586  Edges: 8585
  TF nodes: 1   Gene nodes: 8585

Edges:
  ENCODE-hg38_peaks -> SAMD11   weight=4.96  distance=629bp  qvalue=5.01
  ENCODE-hg38_peaks -> KLHL17   weight=7.03  distance=59bp  qvalue=10.48
  ENCODE-hg38_peaks -> PLEKHN1   weight=5.15  distance=267bp  qvalue=6.53
  ENCODE-hg38_peaks -> PERM1   weight=3.57  distance=6250bp  qvalue=2.31
  ENCODE-hg38_peaks -> HES4   weight=20.32  distance=91bp  qvalue=205.47
  ENCODE-hg38_peaks -> ISG15   weight=3.82  distance=3969bp  qvalue=4.11
  ENCODE-hg38_peaks -> AGRN   weight=6.79  distance=21223bp  qvalue=8.88
  ENCODE-hg38_peaks -> RNF223   weight=5.44  distance=5933bp  qvalue=7.71
  ENCODE-hg38_peaks -> TTLL10   weight=5.11  distance=733bp  qvalue=4.95
  ENCODE-hg38_peaks -> TNFRSF18   weight=9.81  distance=5216bp  qvalue=17.55
  ENCODE-hg38_peaks -> TNFRSF4   weight=3.77  distance=8834bp  qvalue=2.23
  ENCODE-hg38_peaks -> B3GALT6   weight=2.68  distance=232bp  qvalue=1.85
  ENCODE-hg38_peaks -> C1QTNF12   weight=5.28  di

In [10]:
# save for analysis
output_path = "../results/networks/2026-07-31_grn"
#nx.write_gexf(G, output_path + ".gexf")
nx.write_graphml(G, output_path + ".graphml")
print("\nSaved graph")



Saved graph


### **Step 6. Make graph again after filtering with ML model**


In [ ]:
G = build_graph(
    narrowpeak_dir="../results/peaks_filtered/", # folder with narrowpeak files
    tss_bed_path=tss_bed_path,
    qvalue_threshold=1.3 # needed?
)

print(f"Nodes: {G.number_of_nodes()}  Edges: {G.number_of_edges()}")

tf_nodes = [n for n, d in G.nodes(data=True) if d["node_type"] == "TF"]
gene_nodes = [n for n, d in G.nodes(data=True) if d["node_type"] == "gene"]
print(f"  TF nodes: {len(tf_nodes)}   Gene nodes: {len(gene_nodes)}")
